In [11]:
# Run once to make sure everything is installed
# You can skip this cell if your environment is already set up
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'requests', 'pandas', 'nltk'])

0

In [12]:
import requests
import json
import re
import time
import pandas as pd

import nltk
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize


In [13]:
# ---------------Configuration

# Steam App IDs for popular games (change or add more as needed)
# Find an app ID by visiting a store page: store.steampowered.com/app/{appid}
GAME_IDS = {
    730:   'Counter-Strike 2',
    570:   'Dota 2',
    1091500: 'Cyberpunk 2077',
}

MAX_REVIEWS_PER_GAME = 500
REVIEWS_PER_REQUEST  = 100
DELAY_BETWEEN_CALLS  = 1.0   # seconds - be polite to the API
OUTPUT_CSV           = 'steam_reviews_raw.csv'

In [14]:
def fetch_reviews_for_game(appid: int, game_name: str, max_reviews: int = 500) -> list[dict]:
    """
    Fetches up to `max_reviews` reviews for a Steam game using cursor-based pagination.

    Returns a list of dicts, each containing:
        review_text, voted_up, timestamp_created, appid, game_name
    """
    base_url = f'https://store.steampowered.com/appreviews/{appid}'
    params = {
        'json':           1,
        'language':       'english',
        'review_type':    'all',        # 'positive', 'negative', 'all'
        'purchase_type':  'all',
        'num_per_page':   REVIEWS_PER_REQUEST,
        'cursor':         '*',          # start cursor - Steam uses '*' for first page
        'filter':         'recent',     # 'recent','updated', 'all'
    }

    collected = []
    seen_cursors = set()

    while len(collected) < max_reviews:
        try:
            response = requests.get(base_url, params=params, timeout=10)
            response.raise_for_status()
            data = response.json()
        except requests.RequestException as e:
            print(f'   Request error for appid {appid}: {e}')
            break

        if data.get('success') != 1:
            print(f'   API returned success=0 for appid {appid}')
            break

        reviews = data.get('reviews', [])
        if not reviews:
            print(f'   No more reviews available for {game_name}.')
            break

        for r in reviews:
            collected.append({
                'review_text':       r.get('review', ''),
                'voted_up':          r.get('voted_up', None),       # True = recommended
                'timestamp_created': r.get('timestamp_created', None),
                'appid':             appid,
                'game_name':         game_name,
            })

        new_cursor = data.get('cursor', '')
        print(f'  Fetched {len(collected):>4} reviews so far for {game_name}...')

        # Stop if cursor hasn't changed (end of results)
        if new_cursor in seen_cursors or new_cursor == params['cursor']:
            print(f'  [✓] Cursor loop detected - stopping for {game_name}.')
            break

        seen_cursors.add(new_cursor)
        params['cursor'] = new_cursor
        time.sleep(DELAY_BETWEEN_CALLS)

    return collected[:max_reviews]

In [15]:
# --------------- Collect reviews for all configured games
all_reviews = []

for appid, name in GAME_IDS.items():
    print(f'\nFetching reviews for: {name} (appid={appid})')
    reviews = fetch_reviews_for_game(appid, name, max_reviews=MAX_REVIEWS_PER_GAME)
    all_reviews.extend(reviews)
    print(f'  Done. Total reviews collected so far: {len(all_reviews)}')

print(f'\nTotal reviews collected: {len(all_reviews)}')


Fetching reviews for: Counter-Strike 2 (appid=730)
  Fetched  100 reviews so far for Counter-Strike 2...
  Fetched  200 reviews so far for Counter-Strike 2...
  Fetched  300 reviews so far for Counter-Strike 2...
  Fetched  400 reviews so far for Counter-Strike 2...
  Fetched  500 reviews so far for Counter-Strike 2...
  Done. Total reviews collected so far: 500

Fetching reviews for: Dota 2 (appid=570)
  Fetched  100 reviews so far for Dota 2...
  Fetched  200 reviews so far for Dota 2...
  Fetched  300 reviews so far for Dota 2...
  Fetched  400 reviews so far for Dota 2...
  Fetched  500 reviews so far for Dota 2...
  Done. Total reviews collected so far: 1000

Fetching reviews for: Cyberpunk 2077 (appid=1091500)
  Fetched  100 reviews so far for Cyberpunk 2077...
  Fetched  200 reviews so far for Cyberpunk 2077...
  Fetched  300 reviews so far for Cyberpunk 2077...
  Fetched  400 reviews so far for Cyberpunk 2077...
  Fetched  500 reviews so far for Cyberpunk 2077...
  Done. Total

In [32]:
# ------------- Convert to DataFrame and inspect
df_raw = pd.DataFrame(all_reviews)

# Convert Unix timestamp to a readable datetime
df_raw['date'] = pd.to_datetime(df_raw['timestamp_created'], unit='s', errors='coerce')

print(f'Shape: {df_raw.shape}')
print(f'\nSentiment distribution:')
print(df_raw['voted_up'].value_counts().rename({True: 'Positive reviews', False: 'Negative reviews'}).to_string())
print(f'\nReviews per game:')
print(df_raw['game_name'].value_counts().to_string())

df_raw.head(3)

Shape: (1500, 6)

Sentiment distribution:
voted_up
Positive reviews    1246
Negative reviews     254

Reviews per game:
game_name
Counter-Strike 2    500
Dota 2              500
Cyberpunk 2077      500


,review_text,voted_up,timestamp_created,appid,game_name,date
0,Копия Стендофа 2,True,1777153632,730,Counter-Strike 2,2026-04-25 21:47:12
1,BEST GAME,True,1777153455,730,Counter-Strike 2,2026-04-25 21:44:15
2,.,True,1777153357,730,Counter-Strike 2,2026-04-25 21:42:37


In [17]:
# ----------- Drop rows with empty review text
before = len(df_raw)
df_raw = df_raw[df_raw['review_text'].str.strip().str.len() > 0].reset_index(drop=True)
print(f'Removed {before - len(df_raw)} empty reviews. Remaining: {len(df_raw)}')

Removed 5 empty reviews. Remaining: 1495


In [18]:
# ----------- Save raw dataset
df_raw.to_csv(OUTPUT_CSV, index=False)
print(f'Raw dataset saved to: {OUTPUT_CSV}')

Raw dataset saved to: steam_reviews_raw.csv


In [19]:
#  Load from CSV (skip this cell if df_raw is already in memory)
# df_raw = pd.read_csv('steam_reviews_raw.csv')
# df_raw['date'] = pd.to_datetime(df_raw['date'], errors='coerce')
# print(f'Loaded {len(df_raw)} reviews from CSV.')

In [20]:
# ------------- Stopwords list
STOP_WORDS = set(stopwords.words('english'))

# Optional: add game-specific noise words that add no signal
CUSTOM_STOPWORDS = {'game', 'games', 'play', 'played', 'player', 'steam',
                    'get', 'got', 'one', 'would', 'really', 'also', 'like'}
STOP_WORDS.update(CUSTOM_STOPWORDS)

print(f'Total stopwords: {len(STOP_WORDS)}')

Total stopwords: 211


In [21]:
def preprocess_review(text: str) -> str:
    """
    Clean a raw review string and return a preprocessed string of tokens.
    Steps:
        1. Lowercase
        2. Strip URLs
        3. Remove non-ASCII characters
        4. Remove punctuation and digits
        5. Tokenize
        6. Remove stopwords and short tokens
        7. Join into a single clean string
    """
    if not isinstance(text, str) or not text.strip():
        return ''

    #Lowercase
    text = text.lower()

    #Remove URLs
    text = re.sub(r'https?://\S+|www\.\S+', '', text)

    #Remove non-ASCII (emoji, foreign chars)
    text = text.encode('ascii', errors='ignore').decode()

    #Remove punctuation and digits -keep only letters and spaces
    text = re.sub(r'[^a-z\s]', ' ', text)

    # Tokenize
    tokens = word_tokenize(text)

    #Remove stopwords and very short tokens
    tokens = [t for t in tokens if t not in STOP_WORDS and len(t) > 1]

    # 7. Rejoin
    return ' '.join(tokens)

In [22]:
# --------------- Quick sanity check
sample = "10/10 would NOT recommend!!!!! The game crashed 5 times. https://store.steam.com"
print('Before:', sample)
print('After: ', preprocess_review(sample))

Before: 10/10 would NOT recommend!!! The game crashed 5 times. https://store.steam.com
After:  recommend crashed times


In [23]:
#--------------- Apply preprocessing to entire dataset
print('Preprocessing reviews....(this may take time)')

df_raw['review_clean'] = df_raw['review_text'].apply(preprocess_review)

# Drop rows where preprocessing produced an empty string
before = len(df_raw)
df_clean = df_raw[df_raw['review_clean'].str.strip().str.len() > 0].reset_index(drop=True)
print(f'Dropped {before - len(df_clean)} empty rows after preprocessing.')
print(f'Final dataset shape: {df_clean.shape}')

Preprocessing reviews... (this may take a moment)
Dropped 198 empty rows after preprocessing.
Final dataset shape: (1297, 7)


In [24]:
# --------------- Preview some examples
pd.set_option('display.max_colwidth', 120)
df_clean[['review_text', 'review_clean', 'voted_up']].head(5)

,review_text,review_clean,voted_up
0,BEST GAME,best,True
1,runs realy bad bring back 120 tick rate,runs realy bad bring back tick rate,False
2,very good very nice very good very nice,good nice good nice,True
3,very good.,good,True
4,Nice\n,nice,True


In [36]:
#-------------- Basic token statistics
df_clean['token_count'] = df_clean['review_clean'].str.split().str.len()

print('Token count statistics:')
print(df_clean['token_count'].describe().round(2).to_string())
print(f'\nReviews with < 5 tokens: {(df_clean["token_count"] < 5).sum()}')

Token count statistics:
count    512.00
mean      21.15
std       36.81
min        5.00
25%        6.75
50%       10.00
75%       21.00
max      466.00

Reviews with < 5 tokens: 0


In [26]:
# --------------- Optional: filter out very short reviews
MIN_TOKENS = 5
before = len(df_clean)
df_clean = df_clean[df_clean['token_count'] >= MIN_TOKENS].reset_index(drop=True)
print(f'Kept {len(df_clean)} reviews with >= {MIN_TOKENS} tokens (removed {before - len(df_clean)}).')

Kept 512 reviews with >= 5 tokens (removed 785).


In [27]:
# --------------- Save preprocessed dataset
CLEAN_CSV = 'steam_reviews_clean.csv'
df_clean.to_csv(CLEAN_CSV, index=False)
print(f'Preprocessed dataset saved to: {CLEAN_CSV}')
print(f'Columns: {list(df_clean.columns)}')

Preprocessed dataset saved to: steam_reviews_clean.csv
Columns: ['review_text', 'voted_up', 'timestamp_created', 'appid', 'game_name', 'date', 'review_clean', 'token_count']
